# 04 - Pipeline CamemBERT : préparation et entraînement contrôlé

Ce notebook est le point d'entrée principal pour préparer puis lancer le fine-tuning CamemBERT.

Il suit l'ordre suivant : configuration, vérification de l'environnement, contrôle qualité du dataset, préparation du train propre, validation des splits, puis entraînement contrôlé.

## 1. Configuration utilisateur

Les chemins restent dans l'architecture existante du projet : `data/processed`, `models`, `reports`, `src`.

In [ ]:
from pathlib import Path
import sys
import json
import importlib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Recharge les modules locaux si le kernel Jupyter avait gardé une ancienne version.
for module_name in [
    "config.label_schema",
    "src.ml.camembert_trainer",
    "src.ml.dataset_quality",
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from config.label_schema import LABEL_ORDER
from src.ml.camembert_trainer import (
    CamembertTrainingConfig,
    check_training_environment,
    load_csv_split,
    train_camembert,
)
from src.ml.dataset_quality import (
    out_of_scope_topic_issues,
    quality_summary,
    write_train_without_out_of_scope_topic_issues,
)

EXECUTION_MODE = "inspect_only"  # "inspect_only" ou "train_camembert_now"
INSTALL_MISSING_DEPENDENCIES = False
USE_CLEAN_TRAIN = True
CREATE_CLEAN_TRAIN_IF_NEEDED = True

RAW_TRAIN_PATH = ROOT / "data" / "processed" / "train_augmented.csv"
CLEAN_TRAIN_PATH = ROOT / "data" / "processed" / "train_camembert_clean.csv"
VAL_PATH = ROOT / "data" / "processed" / "val.csv"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"
MODEL_DIR = ROOT / "models" / "camembert_classifier"
REPORTS_DIR = ROOT / "reports" / "camembert"

training_config = CamembertTrainingConfig(
    model_name="camembert-base",
    max_length=192,
    seed=42,
    learning_rate=2e-5,
    epochs=6,
    train_batch_size=8,
    eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    warmup_ratio=0.10,
    early_stopping_patience=2,
    use_weighted_loss=True,
)

print("Mode:", EXECUTION_MODE)
print("Raw train:", RAW_TRAIN_PATH)
print("Clean train:", CLEAN_TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)
print("Model output:", MODEL_DIR)
print("Reports:", REPORTS_DIR)


c:\Users\223114186\Documents\chatbot_ia\icu_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mode: train_camembert_now
Raw train: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\train_augmented.csv
Clean train: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\train_camembert_clean.csv
Validation: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\val.csv
Test: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\test.csv
Model output: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\camembert_classifier
Reports: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\camembert


## 2. Vérification de l'environnement

Cette étape évite l'erreur `Trainer requires accelerate`. Si une dépendance manque, active `INSTALL_MISSING_DEPENDENCIES = True`, exécute cette cellule, puis redémarre le kernel avant l'entraînement.

In [ ]:
try:
    versions = check_training_environment()
    print("Environnement OK")
    display(pd.DataFrame([versions]))
except RuntimeError as exc:
    print(exc)
    if INSTALL_MISSING_DEPENDENCIES:
        %pip install accelerate safetensors tensorboard
        print("Installation terminée. Redémarre le kernel, puis relance le notebook.")
    else:
        print("Pour corriger depuis le notebook : mets INSTALL_MISSING_DEPENDENCIES = True, relance cette cellule, puis redémarre le kernel.")


Environnement OK


,torch,transformers,datasets,accelerate,cuda_available,device
0,2.12.0,5.10.2,5.0.0,1.14.0,False,cpu


## 3. Contrôle qualité du train augmenté

Le label `hors_perimetre` doit correspondre à des demandes hors périmètre. Les longs textes médicaux publics mal labellisés comme `hors_perimetre` sont du bruit pour CamemBERT et dégradent le fine-tuning.

In [ ]:
raw_train_df = pd.read_csv(RAW_TRAIN_PATH)
raw_quality = quality_summary(raw_train_df)
issues = out_of_scope_topic_issues(raw_train_df)

display(pd.DataFrame([raw_quality]))
print("Exemples suspects label=hors_perimetre avec topic médical:")
display(issues.head(10))


,rows,out_of_scope_topic_issues,out_of_scope_topic_issue_rate
0,13688,281,0.020529


Exemples suspects label=hors_perimetre avec topic m?dical:


,text,label,topic,source
60,La coagulopathie intra-vasculaire disséminée (...,hors_perimetre,medication,mediqal
62,Compte rendu d'hospitalisation du service de N...,hors_perimetre,oxygen_saturation,parhaf
105,Compte rendu d'hospitalisation\n\nService de M...,hors_perimetre,monitoring,parhaf
115,Compte rendu d'hospitalisation du service de C...,hors_perimetre,lab_results,parhaf
159,Le médecin appelé appartient aux Groupes Régio...,hors_perimetre,communication_with_team,mediqal
195,Compte rendu d'hospitalisation du service de M...,hors_perimetre,lab_results,parhaf
256,(cochez la réponse juste) L'asthme aigu grave ...,hors_perimetre,medication,mediqal
260,Compte rendu d'hospitalisation du service de P...,hors_perimetre,lab_results,parhaf
328,Compte rendu d'hospitalisation du service de C...,hors_perimetre,lab_results,parhaf
336,Compte rendu d'hospitalisation\n\nService de N...,hors_perimetre,monitoring,parhaf


## 4. Préparation du train CamemBERT

Si `USE_CLEAN_TRAIN = True`, le notebook utilise `data/processed/train_camembert_clean.csv`. Ce fichier est créé seulement s'il n'existe pas et si `CREATE_CLEAN_TRAIN_IF_NEEDED = True`.

In [ ]:
if USE_CLEAN_TRAIN:
    if not CLEAN_TRAIN_PATH.exists():
        if CREATE_CLEAN_TRAIN_IF_NEEDED:
            cleaned_train_df = write_train_without_out_of_scope_topic_issues(
                RAW_TRAIN_PATH,
                CLEAN_TRAIN_PATH,
            )
            print("Train nettoyé créé:", CLEAN_TRAIN_PATH)
        else:
            raise FileNotFoundError(
                f"Train nettoyé absent: {CLEAN_TRAIN_PATH}. Active CREATE_CLEAN_TRAIN_IF_NEEDED ou désactive USE_CLEAN_TRAIN."
            )
    TRAIN_PATH = CLEAN_TRAIN_PATH
else:
    TRAIN_PATH = RAW_TRAIN_PATH

print("Train utilisé:", TRAIN_PATH)
train_preview = pd.read_csv(TRAIN_PATH)
display(pd.DataFrame([quality_summary(train_preview)]))


Train utilis?: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\train_camembert_clean.csv


,rows,out_of_scope_topic_issues,out_of_scope_topic_issue_rate
0,13407,0,0.0


## 5. Validation des splits

On vérifie que les fichiers existent, que les colonnes nécessaires sont présentes et que les labels appartiennent au schéma officiel.

In [6]:
train_df = load_csv_split(TRAIN_PATH, "train")
val_df = load_csv_split(VAL_PATH, "validation")
test_df = load_csv_split(TEST_PATH, "test")

summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "path": str(TRAIN_PATH)},
        {"split": "validation", "rows": len(val_df), "path": str(VAL_PATH)},
        {"split": "test", "rows": len(test_df), "path": str(TEST_PATH)},
    ]
)
display(summary)

,split,rows,path
0,train,13407,C:\Users\223114186\Documents\chatbot_ia\icu_ch...
1,validation,1614,C:\Users\223114186\Documents\chatbot_ia\icu_ch...
2,test,1614,C:\Users\223114186\Documents\chatbot_ia\icu_ch...


## 6. Distribution des labels

Cette étape vérifie si l'augmentation garde une distribution exploitable et si les classes critiques restent visibles.

In [7]:
label_dist = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int).reindex(LABEL_ORDER)

label_dist["train_pct"] = (label_dist["train"] / label_dist["train"].sum()).round(4)
label_dist["validation_pct"] = (label_dist["validation"] / label_dist["validation"].sum()).round(4)
label_dist["test_pct"] = (label_dist["test"] / label_dist["test"].sum()).round(4)

display(label_dist)

,train,validation,test,train_pct,validation_pct,test_pct
label,,,,,,
information_generale,2570,321,321,0.1917,0.1989,0.1989
sensible_reformulable,4749,577,576,0.3542,0.3575,0.3569
avis_medical_personnalise,3970,479,479,0.2961,0.2968,0.2968
urgence,1758,202,203,0.1311,0.1252,0.1258
hors_perimetre,360,35,35,0.0269,0.0217,0.0217


## 7. Distribution des topics

Contrôle spécifique : les exemples `hors_perimetre` du train nettoyé ne doivent plus venir de topics médicaux. Les mêmes cas dans validation/test sont affichés comme audit des splits figés, sans nettoyage automatique.

In [ ]:
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"--- {split_name} ---")
    topic_table = pd.crosstab(df["label"], df["topic"]).reindex(index=LABEL_ORDER).fillna(0).astype(int)
    display(topic_table)

print("Anomalies train après préparation:", len(out_of_scope_topic_issues(train_df)))
print("Audit validation figée:", len(out_of_scope_topic_issues(val_df)), "cas hors_perimetre avec topic médical")
print("Audit test figé:", len(out_of_scope_topic_issues(test_df)), "cas hors_perimetre avec topic médical")


--- train ---


topic,coma,communication_with_team,delirium,dialysis,end_of_life,hors_perimetre,imaging,infection,intubation,lab_results,medication,monitoring,oxygen_saturation,prognosis,sedation,sepsis,surgery,ventilation,visits
label,,,,,,,,,,,,,,,,,,,
information_generale,135,155,44,7,0,0,370,778,5,74,712,136,38,0,3,24,31,30,28
sensible_reformulable,78,305,93,245,119,0,706,794,33,631,699,212,328,88,41,39,205,37,96
avis_medical_personnalise,24,12,3,180,122,0,795,112,21,577,1576,65,23,287,45,51,69,5,3
urgence,13,36,11,18,39,0,136,49,17,448,121,257,432,3,32,72,31,37,6
hors_perimetre,0,0,0,0,0,360,0,0,0,0,0,0,0,0,0,0,0,0,0


--- validation ---


topic,coma,communication_with_team,delirium,dialysis,end_of_life,imaging,infection,intubation,lab_results,medication,monitoring,oxygen_saturation,prognosis,sedation,sepsis,surgery,ventilation,visits
label,,,,,,,,,,,,,,,,,,
information_generale,18,20,2,0,0,55,101,1,14,70,19,8,0,1,1,2,2,7
sensible_reformulable,12,15,8,28,11,99,85,4,81,107,19,36,8,8,7,32,7,10
avis_medical_personnalise,2,2,0,29,14,80,17,1,92,186,5,2,24,5,6,12,2,0
urgence,4,3,1,1,2,11,6,3,61,14,19,49,0,3,12,9,2,2
hors_perimetre,0,0,0,0,0,2,1,0,16,5,0,10,0,0,0,0,1,0


--- test ---


topic,coma,communication_with_team,delirium,dialysis,end_of_life,imaging,infection,intubation,lab_results,medication,monitoring,oxygen_saturation,prognosis,sedation,sepsis,surgery,ventilation,visits
label,,,,,,,,,,,,,,,,,,
information_generale,16,15,2,0,0,43,99,0,19,92,15,4,0,1,4,6,0,5
sensible_reformulable,11,20,12,30,9,84,101,4,93,100,17,42,11,5,3,18,9,7
avis_medical_personnalise,0,2,0,22,10,100,23,1,82,180,6,3,27,6,10,6,1,0
urgence,2,3,1,3,3,11,6,7,57,17,22,50,0,2,9,2,7,1
hors_perimetre,0,1,0,1,0,3,5,0,9,6,2,6,1,1,0,0,0,0


Anomalies train apr?s pr?paration: 0
Audit validation fig?e: 35 cas hors_perimetre avec topic m?dical
Audit test fig?: 35 cas hors_perimetre avec topic m?dical


## 8. Hyperparamètres retenus

Configuration volontairement forte mais raisonnable : weighted loss, early stopping, batch effectif par accumulation de gradients, métrique principale `macro_f1`.

In [9]:
display(pd.DataFrame([training_config.__dict__]))

,model_name,max_length,seed,learning_rate,epochs,train_batch_size,eval_batch_size,gradient_accumulation_steps,weight_decay,warmup_ratio,early_stopping_patience,metric_for_best_model,use_weighted_loss
0,camembert-base,192,42,0.00002,6,8,16,2,0.01,0.1,2,macro_f1,True


## 9. Entraînement contrôlé

Cette cellule est la seule qui peut entraîner le modèle. Elle ne fait rien tant que `EXECUTION_MODE` n'est pas exactement `train_camembert_now`.

In [ ]:
if EXECUTION_MODE == "train_camembert_now":
    trainer, val_metrics, test_metrics = train_camembert(
        train_path=TRAIN_PATH,
        val_path=VAL_PATH,
        test_path=TEST_PATH,
        output_dir=MODEL_DIR,
        reports_dir=REPORTS_DIR,
        config=training_config,
    )
    print("Validation metrics")
    print(json.dumps(val_metrics, indent=2, ensure_ascii=False))
    print("Test metrics")
    print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
else:
    print("Entraînement non lancé.")
    print('Pour entraîner depuis ce notebook, mets EXECUTION_MODE = "train_camembert_now" dans la cellule de configuration.')


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2205.18it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio 

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Recall,Weighted F1,Critical Errors,Recall Urgence,Recall Avis Medical Personnalise,Recall Hors Perimetre
1,1.067856,1.049441,0.771375,0.633515,0.653341,0.760057,114,0.871287,0.851775,0.000000
2,0.590568,1.113374,0.820942,0.663913,0.693695,0.813646,80,0.960396,0.858038,0.000000
3,0.554374,1.192136,0.856877,0.693414,0.711642,0.847483,86,0.900990,0.899791,0.000000
4,0.416967,1.267278,0.858116,0.696105,0.710432,0.848627,87,0.891089,0.906054,0.000000
5,0.418461,1.341066,0.861834,0.699185,0.713735,0.852168,82,0.900990,0.912317,0.000000
6,0.315832,1.382229,0.858736,0.697142,0.710353,0.849392,95,0.891089,0.893528,0.000000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
c:\Users\223114186\Documents\chatbot_ia\icu_chatbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]
c:\Users\223114186\Documents\chatbot_ia\icu_chatbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]
c:\Users\223114186\Documents\chatbot_ia\icu_chatbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shard

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Recall,Weighted F1,Critical Errors,Recall Urgence,Recall Avis Medical Personnalise,Recall Hors Perimetre
0.315832,1.341066,6,0.861834,0.699185,0.713735,0.852168,82,0.900990,0.912317,0.000000


c:\Users\223114186\Documents\chatbot_ia\icu_chatbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Recall,Weighted F1,Critical Errors,Recall Urgence,Recall Avis Medical Personnalise,Recall Hors Perimetre
0.315832,1.231235,6,0.876704,0.710144,0.725778,0.866813,75,0.926108,0.918580,0.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Validation metrics
{
  "val_loss": 1.3410656452178955,
  "val_accuracy": 0.8618339529120198,
  "val_macro_f1": 0.6991851489199543,
  "val_macro_recall": 0.7137350909097819,
  "val_weighted_f1": 0.8521681454651938,
  "val_critical_errors": 82,
  "val_recall_urgence": 0.900990099009901,
  "val_recall_avis_medical_personnalise": 0.9123173277661796,
  "val_recall_hors_perimetre": 0.0
}
Test metrics
{
  "test_loss": 1.2312352657318115,
  "test_accuracy": 0.8767038413878563,
  "test_macro_f1": 0.7101438869571091,
  "test_macro_recall": 0.7257783523179511,
  "test_weighted_f1": 0.8668132687517296,
  "test_critical_errors": 75,
  "test_recall_urgence": 0.9261083743842364,
  "test_recall_avis_medical_personnalise": 0.918580375782881,
  "test_recall_hors_perimetre": 0.0
}


## 10. Artefacts attendus

Après entraînement, ces fichiers doivent exister. Le notebook 05 les utilisera pour produire le rapport d'évaluation.

In [11]:
expected_files = [
    MODEL_DIR / "config.json",
    MODEL_DIR / "model.safetensors",
    MODEL_DIR / "tokenizer.json",
    MODEL_DIR / "training_metadata.json",
    MODEL_DIR / "label_mapping.json",
    REPORTS_DIR / "camembert_val_metrics.json",
    REPORTS_DIR / "camembert_test_metrics.json",
]

artifact_status = pd.DataFrame([
    {"artifact": str(path.relative_to(ROOT)), "exists": path.exists()}
    for path in expected_files
])
display(artifact_status)

,artifact,exists
0,models\camembert_classifier\config.json,True
1,models\camembert_classifier\model.safetensors,True
2,models\camembert_classifier\tokenizer.json,True
3,models\camembert_classifier\training_metadata....,True
4,models\camembert_classifier\label_mapping.json,True
5,reports\camembert\camembert_val_metrics.json,True
6,reports\camembert\camembert_test_metrics.json,True
